# Code pour traiter les données collectées par l'application Espace Collaboratif IGN

- Christine Plumejeaud-Perreau, UMR 7301 Migrinter
- 10/07/2026
- Licence AGPL v3

Code déposé dans mon github. 

In [1]:
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

# Extraire des long/lat d'une photo automatiquement

In [2]:


def get_exif_data(image_path):
    image = Image.open(image_path)
    exif_data = {}
    info = image._getexif()

    if info:
        for tag, value in info.items():
            decoded = TAGS.get(tag, tag)
            if decoded == "GPSInfo":
                gps_data = {}
                for t in value:
                    sub_decoded = GPSTAGS.get(t, t)
                    gps_data[sub_decoded] = value[t]
                exif_data["GPSInfo"] = gps_data
            else:
                exif_data[decoded] = value
    return exif_data

def convert_to_degrees(value):
    #'IFDRational' object is not subscriptable
    d, m, s = value
    print(m)
    print(type(m)) #IFDRational
    #https://stackoverflow.com/questions/73105274/python-pil-exif-data-ifdrational-object-does-not-support-indexing

    #return d[0]/d[1] + m[0]/m[1]/60 + s[0]/s[1]/3600
    return float(d) + float(m)/60.0 + float(s)/3600.0

def get_lat_lon(image_path):
    exif = get_exif_data(image_path)
    gps = exif.get("GPSInfo")
    print(gps) #{'GPSVersionID': b'\x02\x02\x00\x00', 'GPSImgDirectionRef': 'M', 'GPSImgDirection': 145.0}
    
    if not gps or 'GPSLatitude' not in gps:
        print("Pas d'info GPS dans mes photos")
        return None, None

    lat = convert_to_degrees(gps["GPSLatitude"])
    if gps["GPSLatitudeRef"] != "N":
        lat = -lat

    lon = convert_to_degrees(gps["GPSLongitude"])
    if gps["GPSLongitudeRef"] != "E":
        lon = -lon

    return lat, lon




### Tests
y'a que pour Brenda que ca marche. 

In [35]:
path = r"C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Photos_22mai2026\PXL_20260521_104429392.jpg"
#path = r"C:\Travail\Enseignement\Cours_M2_python\2026\Photos\PXL_20260626_094630613.jpg"
path = r"E:\Perso\Photos\2020\P_20210110_153902.jpg"
path=r"E:\PIPISTRELLE-juin2019\Perso\Dessin\Bouddha_rieur.jpg"
path=r"D:\Data\IMHANA\Samuel_BUTON\sdrive\Photos_christine_2024\013_carte_Dordogne_Perigord_terrainJuin2024.jpg"
path=r"D:\Data\IMHANA\Samuel_BUTON\sdrive\Périgord_1er semestre 2025_Photos Samuel Buton\Antoine La Grange Abjat en fête\SB1_1005.jpg"
path=r"C:\Travail\MIGRINTER\Admin\Migrinter 2026-06-11 at 23.39.01.jpeg"
path=r"C:\Travail\MIGRINTER\Admin\Velo_Thibaut_2026-06-20 at 21.48.30.jpeg"
path=r"C:\Travail\MIGRINTER\Admin\Lune_2026-06-17 at 22.44.51.jpeg"
path=r"C:\Travail\MIGRINTER\Labo\IMHANA\Terrains\Dordogne\Terrain_2024_Juin2024\Photos\Photo Brenda a trier\IMG_1194.jpg"
# Exemple d'utilisation
lat, lon = get_lat_lon(path)
print("Latitude :", lat)
print("Longitude :", lon)

{'GPSLatitudeRef': 'N', 'GPSLatitude': (45.0, 20.0, 11.28), 'GPSLongitudeRef': 'E', 'GPSLongitude': (1.0, 3.0, 1.67), 'GPSAltitudeRef': b'\x00', 'GPSAltitude': 155.0, 'GPSSpeedRef': 'K', 'GPSSpeed': 0.16618262236551118, 'GPSImgDirectionRef': 'T', 'GPSImgDirection': 134.6097412480974, 'GPSDestBearingRef': 'T', 'GPSDestBearing': 134.6097412480974, 'GPSHPositioningError': 16.002416553390727}
20.0
<class 'PIL.TiffImagePlugin.IFDRational'>
3.0
<class 'PIL.TiffImagePlugin.IFDRational'>
Latitude : 45.33646666666667
Longitude : 1.0504638888888889


### Après activation de la sauvegarde de localisation sur mon appareil, ca marche. 


In [3]:
# https://yesweblog.fr/comment-savoir-si-photo-contient-des-coordonnees-gps/?srsltid=AfmBOoobB29gBi71u0WdSZ7yxXbuVWtfiHPUqcpDyp8DdJmX7wZtUPn6
# Config de mon appareil photo pour enregistrer les long/lat

path=r"C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\PXL_20260710_084208005.jpg"
# Exemple d'utilisation
lat, lon = get_lat_lon(path)
print("Latitude :", lat)
print("Longitude :", lon)

{'GPSVersionID': b'\x02\x02\x00\x00', 'GPSLatitudeRef': 'N', 'GPSLatitude': (46.0, 11.0, 31.91), 'GPSLongitudeRef': 'W', 'GPSLongitude': (0.0, 40.0, 32.87), 'GPSAltitudeRef': b'\x00', 'GPSAltitude': 62.5, 'GPSTimeStamp': (8.0, 41.0, 58.0), 'GPSImgDirectionRef': 'M', 'GPSImgDirection': 297.0, 'GPSDateStamp': '2026:07:10'}
11.0
<class 'PIL.TiffImagePlugin.IFDRational'>
40.0
<class 'PIL.TiffImagePlugin.IFDRational'>
Latitude : 46.19219722222222
Longitude : -0.6757972222222222


# Récupérer tous les signalements IGN Espace Collaboratif, avec leurs photos

Récupérer les photos et faire un geojson en WGS 84, avec des colonnes Theme, Motivation, Appréciation
Fichier encodé en Lamber93



X;Y;identifiant;statut;commentaire;date_ouverture;date_mise_a_jour;date_de_fermeture;attributs;document_1;document_2;document_3;document_4;source;reponse;commune;auteur;pseudo;groupe
591196.581987934;6700014.41298884;1163138;submit;Test;2026/07/02 09:02:11;2026/07/02 09:02:11;;"[{""theme"":""Mes endroits dans Valette"",""attributes"":{""Motivation"":""7. Un endroit que tu aimerais préserver dans l’avenir"",""Appréciation"":""😍""},""groupId"":962}]";;;;;www;;SOINGS-EN-SOLOGNE(41247);3;jmviglino;Valette(962)
589771.625715945;6700699.07866622;1163139;submit;Test;2026/07/02 09:05:19;2026/07/02 09:29:53;;"[{""theme"":""En dehors du quartier, dans la ville et ses extérieurs"",""attributes"":{""Motivation"":""3. Un endroit où tu rêves"",""Appréciation"":""j’adore""},""groupId"":962}]";/document/download/183836;;;;www;;SOINGS-EN-SOLOGNE(41247);3;jmviglino;Valette(962)

[{""theme"":""En dehors du quartier, dans la ville et ses extérieurs"",
    ""attributes"":{""Motivation"":""3. Un endroit où tu rêves"",""Appréciation"":""j’adore""},
    ""groupId"":962}]

Récupérer les photos : 
/document/download/183905
https://espacecollaboratif.ign.fr/document/download/183905

In [ ]:
import pandas as pd
import geopandas as gpd

#Lire "C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\ripart_2026-07-09-07-15-57.csv"
path = r"C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\ripart_2026-07-09-07-15-57.csv"
df = pd.read_csv(path, sep=";")

df.query("date_ouverture >= '08/07/2026'" )



### Filtrer pour n'avoir qu'une partie de la collecte


In [ ]:

from datetime import datetime
date_string = '2026-07-08 12:00:00'
date_objet = datetime.strptime(date_string, '%Y-%m-%d %H:%M:%S')
print(date_objet) # Sortie : 2023-10-30 12:00:00

df['date_ouverture'] = pd.to_datetime(df['date_ouverture'], format='%Y/%m/%d %H:%M:%S', errors='coerce')

#df.loc[df['date_ouverture'] >= date_objet]

df.query(f"date_ouverture >= '{date_objet}'")


2026-07-08 12:00:00


,X,Y,identifiant,statut,commentaire,date_ouverture,date_mise_a_jour,date_de_fermeture,attributs,document_1,document_2,document_3,document_4,source,reponse,commune,auteur,pseudo,groupe
12,433883.236321,6.643245e+06,1165542,submit,Christine au theatre,2026-07-08 13:20:57,2026/07/08 13:20:57,NaN,"[{""theme"":""Mes endroits dans Valette"",""attribu...",/document/download/185017,NaN,NaN,NaN,www,NaN,BRESSUIRE(79049),24565,havalette,Valette(962)
13,433448.344052,6.644375e+06,1165592,submit,Un pique nique a l'ombre est possible,2026-07-08 14:52:34,2026/07/08 14:52:34,NaN,"[{""theme"":""Mes endroits dans Valette"",""attribu...",NaN,NaN,NaN,NaN,www,NaN,BRESSUIRE(79049),24565,havalette,Valette(962)
14,433417.420660,6.644351e+06,1165720,submit,Un parc avec jeux près des abattoirs,2026-07-08 17:07:40,2026/07/08 17:07:40,NaN,"[{""theme"":""Mes endroits dans Valette"",""attribu...",NaN,NaN,NaN,NaN,www,NaN,BRESSUIRE(79049),24565,havalette,Valette(962)
15,433640.600925,6.642422e+06,1165786,submit,"Arrivée à la Valette, entre l'aire d'accueil p...",2026-07-08 21:10:53,2026/07/08 21:10:53,NaN,"[{""theme"":""Mes endroits dans Valette"",""attribu...",/document/download/185065,/document/download/185062,/document/download/185063,/document/download/185064,www,NaN,BRESSUIRE(79049),24565,havalette,Valette(962)


### Décomposer 'attributs' pour récupérer dans des nouvelles colonnes : theme;  Motivation; Appréciation; Note

`Note` est un nouvel attribut que je dérive de l'appréciation. 

Exemple : 
[{""theme"":""En dehors du quartier, dans la ville et ses extérieurs"",
    ""attributes"":{""Motivation"":""3. Un endroit où tu rêves"",""Appréciation"":""j’adore""},
    ""groupId"":962}]

In [ ]:

import json
import pandas as pd

## Associer des notes numériques aux appréciations
dict_notes = {
    'j’aime un peu' : 1,
    'j’adore' : 2,
    'je n’aime pas trop' : -1,
    'je déteste' : -2
}

def parse_attributs(s):
    if pd.isna(s) or not isinstance(s, str) or s.strip()=='':
        return pd.Series([None, None, None, None])
    txt = s
    if txt.startswith('"') and txt.endswith('"'):
        txt = txt[1:-1]
    txt = txt.replace('""', '"')
    try:
        arr = json.loads(txt)
        obj = arr[0] if isinstance(arr, list) and arr else arr
        #Récupère le thème
        theme = obj.get('theme')
        attrs = obj.get('attributes', {}) or {}
        #Récupère la Motivation et l'Appréciation
        mot = attrs.get('Motivation')
        appr = attrs.get('Appréciation')
        # remplacer les émoticônes '😍' dans Appréciation par le texte "j'adore"
        #'😍','❤️','😊','😃','👍','♥','💖', 
        if isinstance(appr, str):
            if any(e in appr for e in ['😍']): 
                appr = "j’adore"
            if any(e in appr for e in ['😉']):
                appr = "j’aime un peu"
            if any(e in appr for e in ['🫣']):
                appr = "je n’aime pas trop"
            if any(e in appr for e in ['😱']):
                appr = "je déteste"
            else:
                appr = appr.strip() or None
        # Traduire les appréciations en notes
        note = dict_notes[appr]
        return pd.Series([theme, mot, appr, note])
    except Exception as e:
        print (e)
        return pd.Series([None, None, None, None])

df[['theme','Motivation','Appréciation', 'Note']] = df['attributs'].apply(parse_attributs)
print(df[['theme','Motivation','Appréciation', 'Note']])

                                                theme  \
0                           Mes endroits dans Valette   
1   En dehors du quartier, dans la ville et ses ex...   
2                           Mes endroits dans Valette   
3                           Mes endroits dans Valette   
4   En dehors du quartier, dans la ville et ses ex...   
5   En dehors du quartier, dans la ville et ses ex...   
6   En dehors du quartier, dans la ville et ses ex...   
7   En dehors du quartier, dans la ville et ses ex...   
8   En dehors du quartier, dans la ville et ses ex...   
9   En dehors du quartier, dans la ville et ses ex...   
10  En dehors du quartier, dans la ville et ses ex...   
11                          Mes endroits dans Valette   
12                          Mes endroits dans Valette   
13                          Mes endroits dans Valette   
14                          Mes endroits dans Valette   
15                          Mes endroits dans Valette   

                              

### Récuperer les photos et les sauver

Récupérer les photos : 
- /document/download/183905
- https://espacecollaboratif.ign.fr/document/download/183905


In [ ]:
import requests
import os.path

#https://fr.python-requests.org/en/latest/user/advanced.html#authentification-personnalisee

stocker = r'C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos'
URL = 'https://espacecollaboratif.ign.fr'

filtre = df.query(f"date_ouverture >= '{date_objet}'")
for index, row in filtre.iterrows():
    for i in [1, 2, 3, 4] : 
        doc = row['document_'+str(i)]
        if not pd.isna(doc) : 
            #print(i)
            fetch_url = URL+doc
            #print(fetch_url)
            dest = os.path.join(stocker, str(row['identifiant']) + '_' + os.path.basename(fetch_url)+'.jpg')
            #print(dest)
            
            resp = requests.get(fetch_url, stream=True, auth=('plumegeo', '********'))
            resp.raise_for_status()

            with open(dest, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            


https://espacecollaboratif.ign.fr/document/download/185017
C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos\1165542_185017
https://espacecollaboratif.ign.fr/document/download/185065
C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos\1165786_185065
https://espacecollaboratif.ign.fr/document/download/185062
C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos\1165786_185062
https://espacecollaboratif.ign.fr/document/download/185063
C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos\1165786_185063
https://espacecollaboratif.ign.fr/document/download/185064
C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\Photos\1165786_185064


### Créer un tableau des identifiants de photos. 

In [64]:
df['photos'] = [[] for _ in range(len(df))]

#filtre = df.query(f"date_ouverture >= '{date_objet}'")
for index, row in df.iterrows():
    photos = []
    for i in [1, 2, 3, 4] : 
        doc = row['document_'+str(i)]
        if not pd.isna(doc) : 
            df.at[index, 'photos'].append(os.path.basename(doc))
            print(i)
df['photos']

1
1
1
1
1
1
1
1
1
1
1
2
1
1
2
3
4


0                                   []
1                             [183836]
2                             [183878]
3                             [183882]
4                             [183904]
5                             [183905]
6                             [183906]
7                             [183907]
8                             [183911]
9                             [183971]
10                            [184590]
11                    [184790, 184790]
12                            [185017]
13                                  []
14                                  []
15    [185065, 185062, 185063, 185064]
Name: photos, dtype: object

### Exporter un geojson avec toutes les infos. 

In [72]:
from shapely.geometry import Point


#filtre = df.query(f"date_ouverture >= '{date_objet}'")
#filtre = df.loc[df['date_ouverture'] >= date_objet]
filtre = df.query(f"date_ouverture >= '{date_objet}'")[['X', 'Y', 'identifiant', 'commentaire', 'date_ouverture', 'commune', 'pseudo', 'photos']]

# Convertir la colonne 'photos' en JSON
filtre['photos'] = filtre['photos'].apply(lambda x: json.dumps(x) if isinstance(x, list) else json.dumps([]))


# créer la géométrie à partir des colonnes X et Y (EPSG:2154 / Lambert-93)
geometry = filtre.apply(lambda r: Point(r['X'], r['Y']), axis=1)
gdf = gpd.GeoDataFrame(filtre, geometry=geometry, crs="EPSG:2154")

# option : reprojeter en WGS84
#gdf_wgs = gdf.to_crs("EPSG:4326")

path = r"C:\Travail\Cartographie\Bressuire_Valette_carto\Préparation_Christine\Macarte_ign_config\collecte_8_juillet_2026\8juillet_2154.geojson"
gdf.to_file(path, driver="GeoJSON")

print(gdf.head())

                X             Y  identifiant  \
12  433883.236321  6.643245e+06      1165542   
13  433448.344052  6.644375e+06      1165592   
14  433417.420660  6.644351e+06      1165720   
15  433640.600925  6.642422e+06      1165786   

                                          commentaire      date_ouverture  \
12                               Christine au theatre 2026-07-08 13:20:57   
13              Un pique nique a l'ombre est possible 2026-07-08 14:52:34   
14               Un parc avec jeux près des abattoirs 2026-07-08 17:07:40   
15  Arrivée à la Valette, entre l'aire d'accueil p... 2026-07-08 21:10:53   

             commune     pseudo                                    photos  \
12  BRESSUIRE(79049)  havalette                                ["185017"]   
13  BRESSUIRE(79049)  havalette                                        []   
14  BRESSUIRE(79049)  havalette                                        []   
15  BRESSUIRE(79049)  havalette  ["185065", "185062", "185063", "

In [73]:
filtre

,X,Y,identifiant,commentaire,date_ouverture,commune,pseudo,photos,geometry
12,433883.236321,6.643245e+06,1165542,Christine au theatre,2026-07-08 13:20:57,BRESSUIRE(79049),havalette,"[""185017""]",POINT (433883.236 6643244.719)
13,433448.344052,6.644375e+06,1165592,Un pique nique a l'ombre est possible,2026-07-08 14:52:34,BRESSUIRE(79049),havalette,[],POINT (433448.344 6644375.315)
14,433417.420660,6.644351e+06,1165720,Un parc avec jeux près des abattoirs,2026-07-08 17:07:40,BRESSUIRE(79049),havalette,[],POINT (433417.421 6644351.496)
15,433640.600925,6.642422e+06,1165786,"Arrivée à la Valette, entre l'aire d'accueil p...",2026-07-08 21:10:53,BRESSUIRE(79049),havalette,"[""185065"", ""185062"", ""185063"", ""185064""]",POINT (433640.601 6642421.981)
